In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import pickle

import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from datetime import datetime
from pandas.tseries.offsets import MonthEnd

input_table = 'TRN_DF_ECOM_OFFTAKE_CHAIN_PSKU'
month_run = '2026-03-31'
os.listdir('/data/aman_singh/acuuracy_check')

['t_s_t_df_ecom_q1.csv',
 'seasonality_all2.csv',
 'chek_nan.csv',
 'acc_framework_all_channels3.xlsx',
 'missing_df_all3.csv',
 'Heuristics_all_combination_ecom_mar_live.xlsx',
 't_thres_df_mt.csv',
 'combine_model+missing_forecasts_brand_asm.ipynb',
 'collate_offtake_heuristics_qcom_city_psku_new_approach_live.ipynb',
 'event_ecom_heuristics2.csv',
 'QCOM Chain PSKU OTP Output',
 'ses_chek.csv',
 "all_channels3 Live Run mar'26.csv",
 'seasonal_p3m_mt.csv',
 'non_co_p3m.csv',
 'ALL Channels Accuracy_fva.ipynb',
 'all_combination_brand_asm_live_heuristics.xlsx',
 'SOH - 01 Feb.xlsx',
 't_thres_df_2.csv',
 'ecom_chain_psku_offtake_to_secondary_v6_PROD.ipynb',
 'seasonality.xlsx',
 'prophet_data_train_till_28_Feb_2026 (7)_ecom.csv',
 'city____.csv',
 'acc_framework_all_channels.xlsx',
 'soh_base_mar_run.csv',
 'chek7.csv',
 'FINAL ECOM Chain PSKU OTP',
 'accuracy_framework3.xlsx',
 'combine_model+missing_forecasts.ipynb',
 'acc_framework_feb.xlsx',
 'handle_missing_forecasts_brand_asm.ip

In [2]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Qcom

In [3]:
qcom_df = pd.read_excel('/data/aman_singh/acuuracy_check/Qcom Depot Psku Primary_Forecast_as_on_12th_march.xlsx', sheet_name = 'Base')
qcom_final = qcom_df[['Depot', 'PSKU', 'Brand', 'Portfolio',
       'Index Rate', 'Run Month', 'Month Date', 'M Month',
       'Calculated Primary Vol', 'Calculated Primary Val','Primary P3M Val', 'LY Primary P3M Val','LY Primary Actuals Val',
       'Primary Actuals Lag 1 Val', 'Primary Actuals Lag 2 Val',
       'Primary Actuals Lag 3 Val']]
qcom_final.columns = qcom_final.columns.str.lower().str.replace(' ', '_')
qcom_final

,depot,psku,brand,portfolio,index_rate,run_month,month_date,m_month,calculated_primary_vol,calculated_primary_val,primary_p3m_val,ly_primary_p3m_val,ly_primary_actuals_val,primary_actuals_lag_1_val,primary_actuals_lag_2_val,primary_actuals_lag_3_val
0,D112,715098,CO_SO_PCP,Skin Care,1220.081000,2026-03-31,2026-04-30,M+1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,D112,715098,CO_SO_PCP,Skin Care,1220.081000,2026-03-31,2026-05-31,M+2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,D112,715098,CO_SO_PCP,Skin Care,1220.081000,2026-03-31,2026-06-30,M+3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,D112,715098,CO_SO_PCP,Skin Care,1220.081000,2026-03-31,2026-07-31,M+4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,D112,715099,CO_SO_PCP,Skin Care,1220.081000,2026-03-31,2026-04-30,M+1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41063,D677,811169,SW_SGPRF,Male Grooming,1443.400363,2026-03-31,2026-07-31,M+4,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
41064,D677,811181,SAF_CDPRS,Saffola Oils,330000.000000,2026-03-31,2026-04-30,M+1,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
41065,D677,811181,SAF_CDPRS,Saffola Oils,330000.000000,2026-03-31,2026-05-31,M+2,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
41066,D677,811181,SAF_CDPRS,Saffola Oils,330000.000000,2026-03-31,2026-06-30,M+3,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN


In [7]:
qcom_final.columns

Index(['depot', 'psku', 'brand', 'portfolio', 'index_rate', 'run_month',
       'month_date', 'm_month', 'calculated_primary_vol',
       'calculated_primary_val', 'primary_p3m_val', 'ly_primary_p3m_val',
       'ly_primary_actuals_val', 'primary_actuals_lag_1_val',
       'primary_actuals_lag_2_val', 'primary_actuals_lag_3_val'],
      dtype='object')

In [4]:
mapping_qcom = pd.read_excel('/data/aman_singh/acuuracy_check/depot_asm_mappings.xlsx', sheet_name = 'Qcom')
qcom_final = qcom_final.merge(mapping_qcom, on = ['depot'], how = 'left')
qcom_final = qcom_final.groupby(['asm_area_code', 'brand','run_month',
       'month_date', 'm_month'])[['calculated_primary_val','primary_p3m_val', 'ly_primary_p3m_val','ly_primary_actuals_val',
       'primary_actuals_lag_1_val', 'primary_actuals_lag_2_val',
       'primary_actuals_lag_3_val']].sum().reset_index()
qcom_final

,asm_area_code,brand,run_month,month_date,m_month,calculated_primary_val,primary_p3m_val,ly_primary_p3m_val,ly_primary_actuals_val,primary_actuals_lag_1_val,primary_actuals_lag_2_val,primary_actuals_lag_3_val
0,QCE1,ADV-AHO-R,2026-03-31,2026-04-30,M+1,0.001326,0.001350,0.001296,0.001215,0.000405,0.002430,0.001215
1,QCE1,ADV-AHO-R,2026-03-31,2026-05-31,M+2,0.001689,0.001350,0.001296,0.001188,0.000405,0.002430,0.001215
2,QCE1,ADV-AHO-R,2026-03-31,2026-06-30,M+3,0.001685,0.001350,0.001296,0.001377,0.000405,0.002430,0.001215
3,QCE1,ADV-AHO-R,2026-03-31,2026-07-31,M+4,0.001724,0.001350,0.001296,0.000405,0.000405,0.002430,0.001215
4,QCE1,BIO OILS,2026-03-31,2026-04-30,M+1,0.006651,0.006363,0.004822,0.002322,0.000557,0.008081,0.010450
...,...,...,...,...,...,...,...,...,...,...,...,...
3147,QCW2,SW_HR_WAX,2026-03-31,2026-07-31,M+4,0.006177,0.006086,0.003768,0.003726,0.007328,0.004223,0.006707
3148,QCW2,SW_SGPRF,2026-03-31,2026-04-30,M+1,0.004348,0.001697,0.007159,0.001109,0.002078,0.000000,0.003014
3149,QCW2,SW_SGPRF,2026-03-31,2026-05-31,M+2,0.004507,0.001697,0.007159,0.001109,0.002078,0.000000,0.003014
3150,QCW2,SW_SGPRF,2026-03-31,2026-06-30,M+3,0.004205,0.001697,0.007159,0.000277,0.002078,0.000000,0.003014


In [5]:
qcom_final.columns = ['asm_area_code', 'brand_code', 'run_month', 'month_date',
       'M month', 'final_pred_value', 'P3M_value', 'LY P3M_value', 'LY value',
       'Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2',
       'Sec_Value_in_Cr_lag_3']
qcom_final['channel'] = 'QCOM'
qcom_final

,asm_area_code,brand_code,run_month,month_date,M month,final_pred_value,P3M_value,LY P3M_value,LY value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,channel
0,QCE1,ADV-AHO-R,2026-03-31,2026-04-30,M+1,0.001326,0.001350,0.001296,0.001215,0.000405,0.002430,0.001215,QCOM
1,QCE1,ADV-AHO-R,2026-03-31,2026-05-31,M+2,0.001689,0.001350,0.001296,0.001188,0.000405,0.002430,0.001215,QCOM
2,QCE1,ADV-AHO-R,2026-03-31,2026-06-30,M+3,0.001685,0.001350,0.001296,0.001377,0.000405,0.002430,0.001215,QCOM
3,QCE1,ADV-AHO-R,2026-03-31,2026-07-31,M+4,0.001724,0.001350,0.001296,0.000405,0.000405,0.002430,0.001215,QCOM
4,QCE1,BIO OILS,2026-03-31,2026-04-30,M+1,0.006651,0.006363,0.004822,0.002322,0.000557,0.008081,0.010450,QCOM
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3147,QCW2,SW_HR_WAX,2026-03-31,2026-07-31,M+4,0.006177,0.006086,0.003768,0.003726,0.007328,0.004223,0.006707,QCOM
3148,QCW2,SW_SGPRF,2026-03-31,2026-04-30,M+1,0.004348,0.001697,0.007159,0.001109,0.002078,0.000000,0.003014,QCOM
3149,QCW2,SW_SGPRF,2026-03-31,2026-05-31,M+2,0.004507,0.001697,0.007159,0.001109,0.002078,0.000000,0.003014,QCOM
3150,QCW2,SW_SGPRF,2026-03-31,2026-06-30,M+3,0.004205,0.001697,0.007159,0.000277,0.002078,0.000000,0.003014,QCOM


### Ecom

In [6]:
ecom_df = pd.read_excel('/data/aman_singh/acuuracy_check/Ecom Depot Psku Primary_as_on_10th_mar.xlsx', sheet_name = 'Base')
ecom_final = ecom_df[['Depot', 'PSKU', 'Brand', 
       'Index Rate', 'Run Month', 'Month Date', 'M Month',
       'Calculated Depot PSKU Primary Vol', 'Calculated Depot PSKU Primary Val','Primary P3M Val', 'LY Primary P3M Val','LY Primary Actuals Val',
       'Primary Actuals Lag 1 Val', 'Primary Actuals Lag 2 Val',
       'Primary Actuals Lag 3 Val']]
ecom_final.columns = ecom_final.columns.str.lower().str.replace(' ', '_')
ecom_final

,depot,psku,brand,index_rate,run_month,month_date,m_month,calculated_depot_psku_primary_vol,calculated_depot_psku_primary_val,primary_p3m_val,ly_primary_p3m_val,ly_primary_actuals_val,primary_actuals_lag_1_val,primary_actuals_lag_2_val,primary_actuals_lag_3_val
0,D111,709567,SAFF OATS,126480.737807,2026-03-31,2026-04-30,M+1,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,D111,709567,SAFF OATS,126480.737807,2026-03-31,2026-05-31,M+2,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,D111,709567,SAFF OATS,126480.737807,2026-03-31,2026-06-30,M+3,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,D111,709567,SAFF OATS,126480.737807,2026-03-31,2026-07-31,M+4,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,D111,718288,SAFF GOLD,137662.938527,2026-03-31,2026-04-30,M+1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39327,D677,811169,SW_SGPRF,1443.400363,2026-03-31,2026-07-31,M+4,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN
39328,D677,811181,SAF_CDPRS,330000.000000,2026-03-31,2026-04-30,M+1,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
39329,D677,811181,SAF_CDPRS,330000.000000,2026-03-31,2026-05-31,M+2,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
39330,D677,811181,SAF_CDPRS,330000.000000,2026-03-31,2026-06-30,M+3,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN


In [7]:
ecom_final.columns

Index(['depot', 'psku', 'brand', 'index_rate', 'run_month', 'month_date',
       'm_month', 'calculated_depot_psku_primary_vol',
       'calculated_depot_psku_primary_val', 'primary_p3m_val',
       'ly_primary_p3m_val', 'ly_primary_actuals_val',
       'primary_actuals_lag_1_val', 'primary_actuals_lag_2_val',
       'primary_actuals_lag_3_val'],
      dtype='object')

In [8]:
mapping_ecom = pd.read_excel('/data/aman_singh/acuuracy_check/depot_asm_mappings.xlsx', sheet_name = 'Ecom')
ecom_final = ecom_final.merge(mapping_ecom, on = ['depot'], how = 'left')
ecom_final = ecom_final.groupby(['asm_area_code', 'brand','run_month',
       'month_date', 'm_month'])[['calculated_depot_psku_primary_val','primary_p3m_val', 'ly_primary_p3m_val','ly_primary_actuals_val',
       'primary_actuals_lag_1_val', 'primary_actuals_lag_2_val',
       'primary_actuals_lag_3_val']].sum().reset_index()
ecom_final

,asm_area_code,brand,run_month,month_date,m_month,calculated_depot_psku_primary_val,primary_p3m_val,ly_primary_p3m_val,ly_primary_actuals_val,primary_actuals_lag_1_val,primary_actuals_lag_2_val,primary_actuals_lag_3_val
0,ECE1,ADV-AHO-R,2026-03-31,2026-04-30,M+1,0.010479,0.010262,0.015951,0.015271,0.010127,0.011828,0.008830
1,ECE1,ADV-AHO-R,2026-03-31,2026-05-31,M+2,0.010609,0.010262,0.017395,0.010667,0.010127,0.011828,0.008830
2,ECE1,ADV-AHO-R,2026-03-31,2026-06-30,M+3,0.010600,0.010262,0.017080,0.009600,0.010127,0.011828,0.008830
3,ECE1,ADV-AHO-R,2026-03-31,2026-07-31,M+4,0.011067,0.010262,0.011846,0.008101,0.010127,0.011828,0.008830
4,ECE1,BIO OILS,2026-03-31,2026-04-30,M+1,0.053752,0.061802,0.062545,0.080859,0.048534,0.061910,0.074961
...,...,...,...,...,...,...,...,...,...,...,...,...
1787,ECW2,SW_HR_WAX,2026-03-31,2026-07-31,M+4,0.004980,0.001994,0.004934,0.003654,0.002484,0.001242,0.002256
1788,ECW2,SW_SGPRF,2026-03-31,2026-04-30,M+1,0.063978,0.018164,0.059087,0.013787,0.019295,0.011778,0.023418
1789,ECW2,SW_SGPRF,2026-03-31,2026-05-31,M+2,0.064720,0.018164,0.046535,0.033048,0.019295,0.011778,0.023418
1790,ECW2,SW_SGPRF,2026-03-31,2026-06-30,M+3,0.054123,0.018164,0.026339,0.027644,0.019295,0.011778,0.023418


In [9]:
ecom_final.columns = ['asm_area_code', 'brand_code', 'run_month', 'month_date',
       'M month', 'final_pred_value', 'P3M_value', 'LY P3M_value', 'LY value',
       'Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2',
       'Sec_Value_in_Cr_lag_3']
ecom_final['channel'] = 'ECOM'
ecom_final

,asm_area_code,brand_code,run_month,month_date,M month,final_pred_value,P3M_value,LY P3M_value,LY value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,channel
0,ECE1,ADV-AHO-R,2026-03-31,2026-04-30,M+1,0.010479,0.010262,0.015951,0.015271,0.010127,0.011828,0.008830,ECOM
1,ECE1,ADV-AHO-R,2026-03-31,2026-05-31,M+2,0.010609,0.010262,0.017395,0.010667,0.010127,0.011828,0.008830,ECOM
2,ECE1,ADV-AHO-R,2026-03-31,2026-06-30,M+3,0.010600,0.010262,0.017080,0.009600,0.010127,0.011828,0.008830,ECOM
3,ECE1,ADV-AHO-R,2026-03-31,2026-07-31,M+4,0.011067,0.010262,0.011846,0.008101,0.010127,0.011828,0.008830,ECOM
4,ECE1,BIO OILS,2026-03-31,2026-04-30,M+1,0.053752,0.061802,0.062545,0.080859,0.048534,0.061910,0.074961,ECOM
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1787,ECW2,SW_HR_WAX,2026-03-31,2026-07-31,M+4,0.004980,0.001994,0.004934,0.003654,0.002484,0.001242,0.002256,ECOM
1788,ECW2,SW_SGPRF,2026-03-31,2026-04-30,M+1,0.063978,0.018164,0.059087,0.013787,0.019295,0.011778,0.023418,ECOM
1789,ECW2,SW_SGPRF,2026-03-31,2026-05-31,M+2,0.064720,0.018164,0.046535,0.033048,0.019295,0.011778,0.023418,ECOM
1790,ECW2,SW_SGPRF,2026-03-31,2026-06-30,M+3,0.054123,0.018164,0.026339,0.027644,0.019295,0.011778,0.023418,ECOM


### GT

In [10]:
gt_df = pd.read_excel("/data/aman_singh/acuuracy_check/Stat Demand Forecast GT_as_on_11th_Mar_2026.xlsb", sheet_name = 'Base')
gt_df

,Channel,Sub Channel,Portfolio,Brand,Brand Class,Qtr Index Rate,Key,Month,ASM,Depot,...,LY Value Lead 2 (Cr),Planning Principle,LY Val (Cr),P3M Val (Cr),P6M Val (Cr),LY P3M Val (Cr),LY P6M Val (Cr),LY P3M (Rolling) Val (Cr),Pred Val (Cr),Primary P3M 0?
0,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,46142,AURG,D3A4,...,0.052587,valid,0.045153,0.060870,0.058300,0.050912,0.046186,0.045543,0.063553,False
1,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,46173,AURG,D3A4,...,0.064426,valid,0.048526,0.060870,0.058300,0.050912,0.044190,0.039830,0.063553,False
2,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,46203,AURG,D3A4,...,0.050040,valid,0.052587,0.060870,0.058300,0.050912,0.047517,0.044121,0.063553,False
3,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,46234,AURG,D3A4,...,0.056373,valid,0.064426,0.060870,0.058300,0.050912,0.047150,0.048756,0.063553,False
4,GT,NaN,CNO,PCNO(R),A,309765.865129,AURG_D3A4_718297,46142,AURG,D3A4,...,1.297083,valid,0.542896,0.700463,0.616672,0.382406,0.356463,0.266461,0.795640,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33967,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_808731,46234,WUP,D113,...,0.000000,valid,0.000129,0.000000,0.000000,0.000129,0.000022,0.000000,0.000000,True
33968,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,46142,WUP,D113,...,0.000000,valid,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False
33969,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,46173,WUP,D113,...,0.000000,valid,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False
33970,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,46203,WUP,D113,...,0.000000,valid,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False


In [11]:
gt_df.columns

Index(['Channel', 'Sub Channel', 'Portfolio', 'Brand', 'Brand Class',
       'Qtr Index Rate', 'Key', 'Month', 'ASM', 'Depot', 'PSKU',
       'LY Vol (ROUM)', 'P3M Vol (ROUM)', 'P6M Vol (ROUM)',
       'LY P3M Vol (ROUM)', 'LY P6M Vol (ROUM)', 'LY P3M (Rolling) Vol (ROUM)',
       'Pred Vol (ROUM)', 'Sec Value Lag 1 (Cr)', 'Sec Value Lag 2 (Cr)',
       'Sec Value Lag 3 (Cr)', 'LY Value Lag 1 (Cr)', 'LY Value Lag 2 (Cr)',
       'LY Value Lead 1 (Cr)', 'LY Value Lead 2 (Cr)', 'Planning Principle',
       'LY Val (Cr)', 'P3M Val (Cr)', 'P6M Val (Cr)', 'LY P3M Val (Cr)',
       'LY P6M Val (Cr)', 'LY P3M (Rolling) Val (Cr)', 'Pred Val (Cr)',
       'Primary P3M 0?'],
      dtype='object')

In [12]:
gt_df['run_month'] = '2026-03-31'
gt_df['Month'] = (
    pd.to_datetime(gt_df['Month'], unit='D', origin='1899-12-30')
      .dt.to_period('M')
      .dt.to_timestamp()
)
gt_df['Month'] = (
    pd.to_datetime(gt_df['Month']) + pd.offsets.MonthEnd(0)
)


In [14]:
gt_df['run_month'] = pd.to_datetime(gt_df['run_month'])
gt_df['Month'] = pd.to_datetime(gt_df['Month'])


mappings = {}

for run_month in gt_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   
gt_df['M month'] = gt_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['Month']
    ), axis=1
)
gt_df

,Channel,Sub Channel,Portfolio,Brand,Brand Class,Qtr Index Rate,Key,Month,ASM,Depot,...,LY Val (Cr),P3M Val (Cr),P6M Val (Cr),LY P3M Val (Cr),LY P6M Val (Cr),LY P3M (Rolling) Val (Cr),Pred Val (Cr),Primary P3M 0?,run_month,M month
0,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,2026-04-30,AURG,D3A4,...,0.045153,0.060870,0.058300,0.050912,0.046186,0.045543,0.063553,False,2026-03-31,M+1
1,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,2026-05-31,AURG,D3A4,...,0.048526,0.060870,0.058300,0.050912,0.044190,0.039830,0.063553,False,2026-03-31,M+2
2,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,2026-06-30,AURG,D3A4,...,0.052587,0.060870,0.058300,0.050912,0.047517,0.044121,0.063553,False,2026-03-31,M+3
3,GT,NaN,Saffola Oils,SAFF GOLD,A,137662.938527,AURG_D3A4_718288,2026-07-31,AURG,D3A4,...,0.064426,0.060870,0.058300,0.050912,0.047150,0.048756,0.063553,False,2026-03-31,M+4
4,GT,NaN,CNO,PCNO(R),A,309765.865129,AURG_D3A4_718297,2026-04-30,AURG,D3A4,...,0.542896,0.700463,0.616672,0.382406,0.356463,0.266461,0.795640,False,2026-03-31,M+1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33967,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_808731,2026-07-31,WUP,D113,...,0.000129,0.000000,0.000000,0.000129,0.000022,0.000000,0.000000,True,2026-03-31,M+4
33968,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,2026-04-30,WUP,D113,...,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False,2026-03-31,M+1
33969,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,2026-05-31,WUP,D113,...,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False,2026-03-31,M+2
33970,GT,NaN,Foods,SAF_HONEY,C,215312.053116,WUP_D113_809238,2026-06-30,WUP,D113,...,0.000000,0.000014,0.000007,0.000000,0.000000,0.000000,0.000000,False,2026-03-31,M+3


In [18]:
gt_df = gt_df.groupby(['ASM', 'Brand', 'run_month','Month', 'M month'])[['Pred Val (Cr)','P3M Val (Cr)', 'LY P3M Val (Cr)',
                                                                 'LY Val (Cr)', 'Sec Value Lag 1 (Cr)', 'Sec Value Lag 2 (Cr)',
                 'Sec Value Lag 3 (Cr)']].sum().reset_index()
gt_df.columns = ['asm_area_code', 'brand_code', 'run_month', 'month_date',
       'M month', 'final_pred_value', 'P3M_value', 'LY P3M_value', 'LY value',
       'Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2',
       'Sec_Value_in_Cr_lag_3']
gt_df['channel'] = 'GT'
gt_df

,asm_area_code,brand_code,run_month,month_date,M month,final_pred_value,P3M_value,LY P3M_value,LY value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,channel
0,AURG,ADV-AHO-R,2026-03-31,2026-04-30,M+1,0.001451,0.000962,0.016435,0.000488,0.000385,0.001611,0.000891,GT
1,AURG,ADV-AHO-R,2026-03-31,2026-05-31,M+2,0.002300,0.000962,0.016435,0.000140,0.000385,0.001611,0.000891,GT
2,AURG,ADV-AHO-R,2026-03-31,2026-06-30,M+3,0.002227,0.000962,0.016435,0.001830,0.000385,0.001611,0.000891,GT
3,AURG,ADV-AHO-R,2026-03-31,2026-07-31,M+4,0.002077,0.000962,0.016435,0.001949,0.000385,0.001611,0.000891,GT
4,AURG,BIO OILS,2026-03-31,2026-04-30,M+1,0.000721,0.000642,0.000952,0.000929,0.000348,0.000534,0.001045,GT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8039,WUP,SW STLDEO,2026-03-31,2026-07-31,M+4,0.000000,0.000000,0.000003,0.000000,0.000000,0.000000,0.000000,GT
8040,WUP,SW_HR_WAX,2026-03-31,2026-04-30,M+1,0.000029,0.000017,0.000085,0.000086,0.000000,0.000026,0.000026,GT
8041,WUP,SW_HR_WAX,2026-03-31,2026-05-31,M+2,0.000118,0.000017,0.000085,0.000290,0.000000,0.000026,0.000026,GT
8042,WUP,SW_HR_WAX,2026-03-31,2026-06-30,M+3,0.000085,0.000017,0.000085,0.000153,0.000000,0.000026,0.000026,GT


### MT

In [19]:
mt_df = pd.read_excel("/data/aman_singh/acuuracy_check/Stat Demand Forecast MT_as_on_11th_Mar_2026.xlsb", sheet_name = 'Base')
mt_df['run_month'] = '2026-03-31'
mt_df['Month'] = (
    pd.to_datetime(mt_df['Month'], unit='D', origin='1899-12-30')
      .dt.to_period('M')
      .dt.to_timestamp()
)
mt_df['Month'] = (
    pd.to_datetime(mt_df['Month']) + pd.offsets.MonthEnd(0)
)

mt_df['run_month'] = pd.to_datetime(mt_df['run_month'])
mt_df['Month'] = pd.to_datetime(mt_df['Month'])


mappings = {}

for run_month in mt_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"

 
mt_df['M month'] = mt_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['Month']
    ), axis=1
)
mt_df = mt_df.groupby(['ASM', 'Brand', 'run_month','Month', 'M month'])[['Pred Val (Cr)','P3M Val (Cr)', 'LY P3M Val (Cr)',
                                                                 'LY Val (Cr)', 'Sec Value Lag 1 (Cr)', 'Sec Value Lag 2 (Cr)',
                 'Sec Value Lag 3 (Cr)']].sum().reset_index()
mt_df.columns = ['asm_area_code', 'brand_code', 'run_month', 'month_date',
       'M month', 'final_pred_value', 'P3M_value', 'LY P3M_value', 'LY value',
       'Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2',
       'Sec_Value_in_Cr_lag_3']
mt_df['channel'] = 'MT'
mt_df

,asm_area_code,brand_code,run_month,month_date,M month,final_pred_value,P3M_value,LY P3M_value,LY value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,channel
0,MCE1,ADV-AHO-R,2026-03-31,2026-04-30,M+1,0.000742,0.000675,0.000621,0.000810,0.000810,0.000405,0.000810,MT
1,MCE1,ADV-AHO-R,2026-03-31,2026-05-31,M+2,0.000767,0.000675,0.000621,0.000810,0.000810,0.000405,0.000810,MT
2,MCE1,ADV-AHO-R,2026-03-31,2026-06-30,M+3,0.000673,0.000675,0.000621,0.000648,0.000810,0.000405,0.000810,MT
3,MCE1,ADV-AHO-R,2026-03-31,2026-07-31,M+4,0.001103,0.000675,0.000621,0.000648,0.000810,0.000405,0.000810,MT
4,MCE1,BIO OILS,2026-03-31,2026-04-30,M+1,0.003152,0.003267,0.003793,0.002880,0.000557,0.004041,0.005202,MT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2167,MCW2,SW_SGPRF,2026-03-31,2026-07-31,M+4,0.000000,0.000000,0.002598,0.001039,0.000000,0.000000,0.000000,MT
2168,MCW2,TRU_ELMNT,2026-03-31,2026-04-30,M+1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,MT
2169,MCW2,TRU_ELMNT,2026-03-31,2026-05-31,M+2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,MT
2170,MCW2,TRU_ELMNT,2026-03-31,2026-06-30,M+3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,MT


In [23]:
qcom_final[qcom_final['M month'] == 'M+1']['final_pred_value'].sum()

30.7497607374193

### ALL

In [24]:
all_channels = pd.read_excel('/data/aman_singh/acuuracy_check/all_combination_brand_asm_live_heuristics.xlsx', sheet_name = 'Base')
all_channels

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value,non_seasonal_growth,Final Heuristic Value2,lyp3m2,ly2
0,BCE1_ADV-AHO-R,2026-03-31,26.40,22.20,11.217182,18.1440,0.001188,0.000999,0.000505,0.000817,...,0.000810,0.000594,1.80,21.60,0.000081,0.000972,2.000000,0.000547,38.40,0.00
1,BCE1_ADV-AHO-R,2026-04-30,26.40,22.20,6.169579,15.9840,0.001188,0.000999,0.000278,0.000719,...,0.000810,0.000594,1.80,21.60,0.000081,0.000972,2.000000,0.000547,38.40,0.00
2,BCE1_ADV-AHO-R,2026-05-31,26.40,22.20,16.809993,16.6740,0.001188,0.000999,0.000757,0.000750,...,0.000810,0.000594,1.80,21.60,0.000081,0.000972,2.000000,0.000547,38.40,14.40
3,BCE1_ADV-AHO-R,2026-06-30,26.40,22.20,13.057868,11.1660,0.001188,0.000999,0.000588,0.000503,...,0.000810,0.000594,1.80,21.60,0.000081,0.000972,2.000000,0.000547,38.40,10.80
4,BCE1_ADV-AHO-R,2026-07-31,26.40,22.20,19.205087,11.8848,0.001188,0.000999,0.000864,0.000535,...,0.000810,0.000594,1.80,21.60,0.000081,0.000972,2.000000,0.000547,38.40,9.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57975,QCW2_SW_SGPRF,2026-11-30,11.76,5.88,0.000000,2.0800,0.001697,0.000849,0.000000,0.000300,...,0.001697,0.000849,49.28,49.28,0.007113,0.007113,0.238636,NaN,49.28,0.00
57976,QCW2_SW_SGPRF,2026-12-31,11.76,5.88,0.000000,10.5840,0.001697,0.000849,0.000000,0.001528,...,0.001697,0.000849,49.28,49.28,0.007113,0.007113,0.238636,NaN,49.28,20.88
57977,QCW2_SW_SGPRF,2027-01-31,11.76,5.88,0.000000,3.2960,0.001697,0.000849,0.000000,0.000476,...,0.001697,0.000849,49.28,49.28,0.007113,0.007113,0.238636,NaN,49.28,0.00
57978,QCW2_SW_SGPRF,2027-02-28,11.76,5.88,0.000000,6.6320,0.001697,0.000849,0.000000,0.000957,...,0.001697,0.000849,49.28,49.28,0.007113,0.007113,0.238636,NaN,49.28,14.40


In [25]:
all_channels.columns[60:]
all_data = all_channels[['channel', 'asm_area_code', 'brand_code','run_month_x','month_date','M month','final_heuristic_60_prophet_value_2',
              'P3M_value', 'LY P3M_value','LY value','Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2',
       'Sec_Value_in_Cr_lag_3']].rename(columns = {'run_month_x':'run_month', 'final_heuristic_60_prophet_value_2':'final_pred_value'})
all_data.columns

Index(['channel', 'asm_area_code', 'brand_code', 'run_month', 'month_date',
       'M month', 'final_pred_value', 'P3M_value', 'LY P3M_value', 'LY value',
       'Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2',
       'Sec_Value_in_Cr_lag_3'],
      dtype='object')

In [29]:
all_data_m4_lat = all_data[all_data['month_date']>'2026-07-31']

### concat all four channels

In [28]:
qcom_final['month_date'] = pd.to_datetime(qcom_final['month_date'])
qcom_final['run_month'] = pd.to_datetime(qcom_final['run_month'])
ecom_final['month_date'] = pd.to_datetime(ecom_final['month_date'])
ecom_final['run_month'] = pd.to_datetime(ecom_final['run_month'])
gt_df['month_date'] = pd.to_datetime(gt_df['month_date'])
gt_df['run_month'] = pd.to_datetime(gt_df['run_month'])
mt_df['month_date'] = pd.to_datetime(mt_df['month_date'])
mt_df['run_month'] = pd.to_datetime(mt_df['run_month'])
final_m4 = pd.concat([qcom_final, ecom_final, gt_df, mt_df], axis = 0)
final_m4

,asm_area_code,brand_code,run_month,month_date,M month,final_pred_value,P3M_value,LY P3M_value,LY value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,channel
0,QCE1,ADV-AHO-R,2026-03-31,2026-04-30,M+1,0.001326,0.001350,0.001296,0.001215,0.000405,0.002430,0.001215,QCOM
1,QCE1,ADV-AHO-R,2026-03-31,2026-05-31,M+2,0.001689,0.001350,0.001296,0.001188,0.000405,0.002430,0.001215,QCOM
2,QCE1,ADV-AHO-R,2026-03-31,2026-06-30,M+3,0.001685,0.001350,0.001296,0.001377,0.000405,0.002430,0.001215,QCOM
3,QCE1,ADV-AHO-R,2026-03-31,2026-07-31,M+4,0.001724,0.001350,0.001296,0.000405,0.000405,0.002430,0.001215,QCOM
4,QCE1,BIO OILS,2026-03-31,2026-04-30,M+1,0.006651,0.006363,0.004822,0.002322,0.000557,0.008081,0.010450,QCOM
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2167,MCW2,SW_SGPRF,2026-03-31,2026-07-31,M+4,0.000000,0.000000,0.002598,0.001039,0.000000,0.000000,0.000000,MT
2168,MCW2,TRU_ELMNT,2026-03-31,2026-04-30,M+1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,MT
2169,MCW2,TRU_ELMNT,2026-03-31,2026-05-31,M+2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,MT
2170,MCW2,TRU_ELMNT,2026-03-31,2026-06-30,M+3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,MT


In [30]:
all_data_m4_lat['month_date'] = pd.to_datetime(all_data_m4_lat['month_date'])
all_data_m4_lat['run_month'] = pd.to_datetime(all_data_m4_lat['run_month'])
final_data = pd.concat([final_m4, all_data_m4_lat], axis = 0)
final_data

,asm_area_code,brand_code,run_month,month_date,M month,final_pred_value,P3M_value,LY P3M_value,LY value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,channel
0,QCE1,ADV-AHO-R,2026-03-31,2026-04-30,M+1,0.001326,0.001350,0.001296,0.001215,0.000405,0.002430,0.001215,QCOM
1,QCE1,ADV-AHO-R,2026-03-31,2026-05-31,M+2,0.001689,0.001350,0.001296,0.001188,0.000405,0.002430,0.001215,QCOM
2,QCE1,ADV-AHO-R,2026-03-31,2026-06-30,M+3,0.001685,0.001350,0.001296,0.001377,0.000405,0.002430,0.001215,QCOM
3,QCE1,ADV-AHO-R,2026-03-31,2026-07-31,M+4,0.001724,0.001350,0.001296,0.000405,0.000405,0.002430,0.001215,QCOM
4,QCE1,BIO OILS,2026-03-31,2026-04-30,M+1,0.006651,0.006363,0.004822,0.002322,0.000557,0.008081,0.010450,QCOM
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57975,QCW2,SW_SGPRF,2026-03-31,2026-11-30,M+8,0.001697,0.001697,0.007113,0.000000,0.002078,0.000000,0.003014,QCOM
57976,QCW2,SW_SGPRF,2026-03-31,2026-12-31,M+9,0.001697,0.001697,0.007113,0.003014,0.002078,0.000000,0.003014,QCOM
57977,QCW2,SW_SGPRF,2026-03-31,2027-01-31,M+10,0.001697,0.001697,0.007113,0.000000,0.002078,0.000000,0.003014,QCOM
57978,QCW2,SW_SGPRF,2026-03-31,2027-02-28,M+11,0.001697,0.001697,0.007113,0.002078,0.002078,0.000000,0.003014,QCOM


In [34]:
final_data

,asm_area_code,brand_code,run_month,month_date,M month,final_pred_value,P3M_value,LY P3M_value,LY value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,channel
0,QCE1,ADV-AHO-R,2026-03-31,2026-04-30,M+1,0.001326,0.001350,0.001296,0.001215,0.000405,0.002430,0.001215,QCOM
1,QCE1,ADV-AHO-R,2026-03-31,2026-05-31,M+2,0.001689,0.001350,0.001296,0.001188,0.000405,0.002430,0.001215,QCOM
2,QCE1,ADV-AHO-R,2026-03-31,2026-06-30,M+3,0.001685,0.001350,0.001296,0.001377,0.000405,0.002430,0.001215,QCOM
3,QCE1,ADV-AHO-R,2026-03-31,2026-07-31,M+4,0.001724,0.001350,0.001296,0.000405,0.000405,0.002430,0.001215,QCOM
4,QCE1,BIO OILS,2026-03-31,2026-04-30,M+1,0.006651,0.006363,0.004822,0.002322,0.000557,0.008081,0.010450,QCOM
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57975,QCW2,SW_SGPRF,2026-03-31,2026-11-30,M+8,0.001697,0.001697,0.007113,0.000000,0.002078,0.000000,0.003014,QCOM
57976,QCW2,SW_SGPRF,2026-03-31,2026-12-31,M+9,0.001697,0.001697,0.007113,0.003014,0.002078,0.000000,0.003014,QCOM
57977,QCW2,SW_SGPRF,2026-03-31,2027-01-31,M+10,0.001697,0.001697,0.007113,0.000000,0.002078,0.000000,0.003014,QCOM
57978,QCW2,SW_SGPRF,2026-03-31,2027-02-28,M+11,0.001697,0.001697,0.007113,0.002078,0.002078,0.000000,0.003014,QCOM


In [57]:
#g = final_data.copy()
final_data = g.copy()

In [44]:
final_data[final_data['month_date']=='2026-05-31']['P3M_value'].sum()

3.3417410823129687

In [79]:
# Columns to fill
fill_columns = ['P3M_value', 'LY P3M_value', 'Sec_Value_in_Cr_lag_1', 
                'Sec_Value_in_Cr_lag_2', 'Sec_Value_in_Cr_lag_3']

# Get M+6 values for each channel
m6_data = (
    final_data[final_data['M month'] == 'M+6']
    [['asm_area_code', 'brand_code', 'channel'] + fill_columns]
    .rename(columns={c: f'{c}_m6' for c in fill_columns})
)
# Merge M+6 values back to all rows based on channel
final_data = final_data.merge(m6_data, on=['channel', 'asm_area_code', 'brand_code'], how='left', suffixes=('', '_m6'))

# Fill columns with M+6 values
for col in fill_columns:
    final_data[col] = final_data[f'{col}_m6']
    final_data = final_data.drop(columns=[f'{col}_m6'])

In [80]:
final_data.groupby(['channel','month_date'])[['P3M_value', 'LY P3M_value', 'Sec_Value_in_Cr_lag_1', 
                'Sec_Value_in_Cr_lag_2', 'Sec_Value_in_Cr_lag_3']].sum().reset_index()

,channel,month_date,P3M_value,LY P3M_value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3
0,ECOM,2026-04-30,35.125711,33.435094,31.999911,33.182685,40.194536
1,ECOM,2026-05-31,35.125711,33.435094,31.999911,33.182685,40.194536
2,ECOM,2026-06-30,35.125711,33.435094,31.999911,33.182685,40.194536
3,ECOM,2026-07-31,35.125711,33.435094,31.999911,33.182685,40.194536
4,ECOM,2026-08-31,41.580116,44.354671,37.866048,38.326276,48.063134
5,ECOM,2026-09-30,41.580116,44.354671,37.866048,38.326276,48.063134
6,ECOM,2026-10-31,41.580116,44.354671,37.866048,38.326276,48.063134
7,ECOM,2026-11-30,41.580116,44.354671,37.866048,38.326276,48.063134
8,ECOM,2026-12-31,41.580116,44.354671,37.866048,38.326276,48.063134
9,ECOM,2027-01-31,41.580116,44.354671,37.866048,38.326276,48.063134


In [81]:
final_data.to_excel('/data/aman_singh/acuuracy_check/final_collated_data_12_months_live2.xlsx', index = False)

In [32]:
final_data['M month'].unique()

array(['M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7', 'M+8', 'M+9',
       'M+10', 'M+11', 'M+12'], dtype=object)

In [76]:
final_data = pd.read_excel('/data/aman_singh/acuuracy_check/final_collated_data_12_months_live.xlsx')
final_data[final_data['month_date']=='2026-05-31']['P3M_value'].sum()

568.0197283922394